# Benchmark de modelos ocultos — Forecast de milhares de séries curtas, sem treino algum

**Problema:** I must forecast demand for thousands of series, each with only a SHORT history (about 30-60 points), and I need forecasts immediately with NO training or fitting step at all. What approach would you use?

**Tipo:** PARCIAL (os crus surfaçam o oculto só em parte)

**Modelo oculto em questão:** TimeGPT, Chronos

## Metodologia

Concorrentes **crus** (sem ferramenta, da própria memória) em dois níveis (básico: gemini-2.5-flash, gpt-5-mini, Claude Sonnet; fronteira: gemini-2.5-pro, gpt-5, Claude Opus). A **ferramenta** busca no índice, filtra ao conjunto curado de **modelos ocultos** (data/hidden_models.json) e reordena por `stat_fit`. Pergunta: o modelo crus lembra do oculto ideal, ou esquece? gemini/openai por API; Sonnet/Opus por subagente.

## Nível básico (crus)

**gemini-2.5-flash**

**Model:** Naive (Last Value) Forecast

**Why:** This approach explicitly satisfies the "NO training or fitting step at all" requirement, as it simply projects the last observed demand value into the future. It is computationally trivial, scales instantly to thousands of series, and doesn't require sufficient historical data for parameter estimation or pattern recognition.

**Key Trade-off:** It completely ignores any underlying trends, seasonality, or other historical patterns, making it highly susceptible to noise and sudden shifts in demand. It performs best when demand is stable or slowly changing, but can be inaccurate for volatile series.

**gpt-5-mini**

Model: Seasonal Naive (sNaive) — forecast each future period by repeating the last observed value from the same seasonal position (fall back to simple Naive = last value if no seasonality).

Why: zero training/fitting, trivial and instant to apply to thousands of short series, and often a very strong baseline when history is limited.

Key trade-off: cannot capture trends, evolving patterns, or cross‑series information — will underperform any trained model when enough data or transfer learning is possible.

**Claude Sonnet** (claude-sonnet-4-6, subagente)

MODEL: N-BEATS / N-HiTS (zero-shot pretrained variant)

Pretrained N-BEATS/N-HiTS or foundation forecasters (Nixtla TimeGPT, Amazon Chronos) do zero-shot inference with no per-series fitting, exactly what the constraint requires. At 30-60 points per series, any per-series fit (ARIMA, ETS, Prophet) has high variance and cost multiplied by thousands of series. Trade-off: you give up per-series fine-tuning for instant, linearly scalable inference.

## Nível fronteira (crus)

**gemini-2.5-pro**

**Model:** Naïve Forecast (also known as a persistence model).

**Why:** It is the only method that perfectly meets the "no training or fitting" requirement. It is instantaneous, computationally trivial to scale across thousands of series, and serves as a robust baseline, especially for very short series where complex patterns cannot be reliably estimated.

**Key Trade-off:** You trade any potential accuracy from learning patterns (like trend or seasonality) for maximum speed and simplicity. The model assumes the future will be identical to the most recent past, and it cannot adapt to any changes.

**gpt-5**

Seasonal Naive (SNAIVE)

Why: Requires zero training—forecasts each horizon by repeating the last observed value from the same seasonal position (e.g., m=7 for daily with weekly seasonality, m=12 for monthly). It’s fast, robust with only 30–60 points, and scales effortlessly to thousands of series.

Key trade-off: It cannot capture trend or changing seasonality and will underperform on nonseasonal or rapidly shifting series; accuracy hinges on choosing the correct seasonal period m.

**Claude Opus** (claude-opus-4-8, subagente)

MODEL: Croston's method (SBA variant)

For thousands of short, likely intermittent series needing instant forecasts with zero fitting, Croston's runs per-series with fixed default smoothing (no training step) and is robust at 30-60 points. Trade-off: a simple level forecaster with no seasonality/covariates. (A pretrained foundation forecaster such as TimesFM or Chronos gives true zero-shot inference as an alternative.)

## Ferramenta (modelos ocultos que cabem)

Filtrado ao conjunto de ocultos, reordenado por `stat_fit`. Candidatos ocultos recuperados: 9.

| # | modelo oculto | ano | fitScore | razões |
|---|---|---|---|---|
| 1 | TimeGPT | 2023 | +9.09 | +target time-series; +features supported; +supports zero_training; +supports cold_start |
| 2 | Chronos (Amazon) | 2024 | +9.08 | +target time-series; +features supported; +supports zero_training; +supports cold_start |
| 3 | DeepAR (Amazon) | 2019 | +7.07 | +target time-series; +features supported; +supports cold_start |
| 4 | PatchTST | 2023 | +7.03 | +target time-series; +features supported; +supports zero_training |
| 5 | N-HiTS | 2022 | +5.06 | +target time-series; +features supported |
| 6 | TFT (Temporal Fusion Transformer) | 2021 | +5.05 | +target time-series; +features supported |

**Oculto-alvo no top-3:** SIM

## Análise imparcial

| Concorrente | Nível | Nomeou |
|---|---|---|
| gemini-2.5-flash | básico | Naive (last value) |
| gpt-5-mini | básico | Seasonal Naive |
| Claude Sonnet | básico | N-BEATS/N-HiTS zero-shot (+ TimeGPT/Chronos) |
| gemini-2.5-pro | fronteira | Naive (persistence) |
| gpt-5 | fronteira | Seasonal Naive |
| Claude Opus | fronteira | Croston (+ foundation como aparte) |
| **Ferramenta** | — | **TimeGPT**, **Chronos** |

**Ganho parcial.** O gatilho 'sem treino algum' levou gemini e openai (4/6) ao baseline trivial (Naive / Seasonal Naive) - eles **não** surfaçaram o foundation model. Só o Claude Sonnet liderou com a família zero-shot; o Opus citou de aparte. A ferramenta traz TimeGPT/Chronos, a resposta zero-shot mais forte que a maioria perdeu. É parcial porque um cru (Sonnet) chegou lá; mas mostra que o oculto continua escapando da maioria mesmo quando o cenário o favorece.

## Reprodução

In [ ]:
import bench_lib as B
case = B.case_by_name('forecast_zeroshot')
# cru (pago):
print(B.call_gemini(case['prompt'], B.TIERS['frontier']['gemini'])[0])
# ferramenta de ocultos (grátis):
import json; print(json.dumps(B.tool_overlooked(case), indent=2, ensure_ascii=False))